In [1]:
import pandas as pd
import numpy as np

print("Library berhasil dipanggil!")

Library berhasil dipanggil!


In [2]:
# Gunakan path yang sudah terbukti berhasil tadi
path_ratings = r"C:\Users\brows\Downloads\skripsi_rio\Skripsi_Hybrid_Rekomendasi\dataset\ml-latest-small\ratings.csv"

df_ratings = pd.read_csv(path_ratings)

# Menampilkan 5 data teratas
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
path_movies = r"C:\Users\brows\Downloads\skripsi_rio\Skripsi_Hybrid_Rekomendasi\dataset\ml-latest-small\movies.csv"

df_movies = pd.read_csv(path_movies)

# Menampilkan 5 data film teratas
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# Cek apakah ada data yang kosong (null)
print("Data Rating Kosong:")
print(df_ratings.isnull().sum())

print("\nData Movie Kosong:")
print(df_movies.isnull().sum())

Data Rating Kosong:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Data Movie Kosong:
movieId    0
title      0
genres     0
dtype: int64


In [5]:
# Membuat pivot table
user_item_matrix = df_ratings.pivot(index='userId', columns='movieId', values='rating')

# Menampilkan sedikit bagian matriks
# NaN artinya user tersebut belum menonton film itu
user_item_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Hitung jumlah total sel di matriks
total_sel = user_item_matrix.size

# Hitung jumlah sel yang ada isinya (bukan NaN)
jumlah_rating = user_item_matrix.notna().sum().sum()

# Hitung jumlah sel yang kosong (NaN)
jumlah_kosong = user_item_matrix.isna().sum().sum()

# Hitung persentase sparsity
sparsity = (jumlah_kosong / total_sel) * 100

print(f"Total Sel Matriks : {total_sel}")
print(f"Total Rating Ada  : {jumlah_rating}")
print(f"Total Sel Kosong  : {jumlah_kosong}")
print(f"Tingkat Sparsity  : {sparsity:.2f}%")

Total Sel Matriks : 5931640
Total Rating Ada  : 100836
Total Sel Kosong  : 5830804
Tingkat Sparsity  : 98.30%


In [7]:
# Menggabungkan df_ratings dengan df_movies berdasarkan 'movieId'
df_combined = pd.merge(df_ratings, df_movies, on='movieId')

# Menampilkan 5 data teratas hasil gabungan
df_combined.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [8]:
# 1. Hitung rata-rata rating dan jumlah rating per film
movie_stats = df_combined.groupby('title').agg({'rating': ['mean', 'count']})
movie_stats.columns = ['avg_rating', 'vote_count']

# 2. Cari film "Populer" (Banyak yang rating)
populer = movie_stats.sort_values(by='vote_count', ascending=False).head(10)

# 3. Cari "Hidden Gems" (Rating > 4.0 tapi yang rating dikit, misal 10-50 orang)
hidden_gems = movie_stats[(movie_stats['vote_count'] >= 10) & (movie_stats['vote_count'] <= 50)]
hidden_gems = hidden_gems.sort_values(by='avg_rating', ascending=False).head(10)

print("--- 10 Film Paling Populer (Mainstream) ---")
print(populer)

print("\n--- 10 Film Hidden Gems (Bagus tapi Sepi Peminat) ---")
print(hidden_gems)

--- 10 Film Paling Populer (Mainstream) ---
                                           avg_rating  vote_count
title                                                            
Forrest Gump (1994)                          4.164134         329
Shawshank Redemption, The (1994)             4.429022         317
Pulp Fiction (1994)                          4.197068         307
Silence of the Lambs, The (1991)             4.161290         279
Matrix, The (1999)                           4.192446         278
Star Wars: Episode IV - A New Hope (1977)    4.231076         251
Jurassic Park (1993)                         3.750000         238
Braveheart (1995)                            4.031646         237
Terminator 2: Judgment Day (1991)            3.970982         224
Schindler's List (1993)                      4.225000         220

--- 10 Film Hidden Gems (Bagus tapi Sepi Peminat) ---
                                               avg_rating  vote_count
title                                  

In [9]:
# Tentukan batas: Rating rata-rata di atas 4.0, tapi yang kasih rating cuma dikit (misal di bawah 30 orang)
# Ini adalah definisi "Hidden Gems" atau "Sepi Peminat tapi Bagus"
underrated_movies = movie_stats[(movie_stats['avg_rating'] >= 4.0) & (movie_stats['vote_count'] < 30)]

# Gabungkan dengan data movie asli untuk melihat genrenya
underrated_list = pd.merge(underrated_movies, df_movies, on='title')

print("--- Daftar Film 'Hidden Gems' yang Bisa Direkomendasikan ---")
display(underrated_list.sort_values(by='avg_rating', ascending=False).head(10))

--- Daftar Film 'Hidden Gems' yang Bisa Direkomendasikan ---


,title,avg_rating,vote_count,movieId,genres
2,'Salem's Lot (2004),5.0,1,27751,Drama|Horror|Mystery|Thriller
10,12 Angry Men (1997),5.0,1,77846,Crime|Drama
12,12 Chairs (1976),5.0,1,141816,Adventure|Comedy
2136,Zeitgeist: Moving Forward (2011),5.0,1,84273,Documentary
720,Galaxy of Terror (Quest) (1981),5.0,1,5746,Action|Horror|Mystery|Sci-Fi
713,Front of the Class (2008),5.0,1,95175,Drama
717,Fugitives (1986),5.0,1,173619,Comedy|Crime
760,Girls About Town (1931),5.0,1,84512,Comedy
768,Go for Zucker! (Alles auf Zucker!) (2004),5.0,1,44851,Comedy
773,"Going Places (Valseuses, Les) (1974)",5.0,1,5088,Comedy|Crime|Drama


In [10]:
# Isi nilai NaN dengan 0
user_item_matrix_filled = user_item_matrix.fillna(0)

# Ubah menjadi matriks numpy agar bisa diproses algoritma
matrix_data = user_item_matrix_filled.values

print("Matriks siap diproses oleh SVD.")

Matriks siap diproses oleh SVD.


In [11]:
from sklearn.decomposition import TruncatedSVD

# n_components adalah jumlah 'faktor laten' atau 'tema tersembunyi'
# Kita coba 20 sebagai awalan (standar riset)
svd = TruncatedSVD(n_components=20, random_state=42)
matrix_terkompresi = svd.fit_transform(matrix_data)

# Rekonstruksi matriks (menebak rating yang kosong)
reconstructed_matrix = svd.inverse_transform(matrix_terkompresi)
df_prediksi = pd.DataFrame(reconstructed_matrix, columns=user_item_matrix.columns, index=user_item_matrix.index)

print("Proses SVD Selesai. Prediksi rating untuk seluruh user-item telah dibuat.")
display(df_prediksi.head())

Proses SVD Selesai. Prediksi rating untuk seluruh user-item telah dibuat.


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,2.789173,1.471277,1.280870,-0.024856,0.254759,1.609531,0.232396,0.115948,0.193227,1.286942,...,-0.009538,-0.008175,-0.010901,-0.010901,-0.009538,-0.010901,-0.009538,-0.009538,-0.009538,-0.033415
2,0.016553,0.010828,-0.012655,0.000473,-0.005400,-0.036313,-0.046721,0.008091,-0.002748,-0.107562,...,0.009905,0.008490,0.011320,0.011320,0.009905,0.011320,0.009905,0.009905,0.009905,0.015023
3,-0.014042,0.047094,0.029815,-0.007728,-0.044938,0.034280,-0.017809,0.012079,-0.005503,0.016433,...,0.000003,0.000003,0.000004,0.000004,0.000003,0.000004,0.000003,0.000003,0.000003,-0.003004
4,2.564164,0.067424,0.220364,0.101337,0.394431,0.752309,0.489290,-0.036030,0.131220,-0.132042,...,0.000655,0.000562,0.000749,0.000749,0.000655,0.000749,0.000655,0.000655,0.000655,-0.006352
5,1.392494,0.968864,0.353202,0.125885,0.498931,0.649854,0.512006,0.151404,0.048942,1.132343,...,-0.000573,-0.000491,-0.000655,-0.000655,-0.000573,-0.000655,-0.000573,-0.000573,-0.000573,0.002053


In [12]:
def rekomendasi_hidden_gems(user_id, num_recommendations=5):
    # 1. Ambil prediksi rating untuk user ini
    user_preds = df_prediksi.loc[user_id].sort_values(ascending=False)
    
    # 2. Ambil daftar film yang SUDAH pernah dirating user (biar gak direkomen lagi)
    sudah_ditonton = user_item_matrix.loc[user_id].dropna().index
    
    # 3. Filter prediksi (hanya yang belum ditonton)
    rekomendasi = user_preds[~user_preds.index.isin(sudah_ditonton)]
    
    # 4. Gabungkan dengan info judul & vote_count (dari cell 8 tadi)
    rekom_df = pd.DataFrame(rekomendasi).reset_index()
    rekom_df.columns = ['movieId', 'prediksi_rating']
    rekom_df = pd.merge(rekom_df, df_movies, on='movieId')
    rekom_df = pd.merge(rekom_df, movie_stats, left_on='title', right_index=True)
    
    # 5. FILTER HIDDEN GEMS: Rating prediksi tinggi (>3.5) tapi vote_count rendah (<50)
    hidden_gems = rekom_df[rekom_df['vote_count'] < 50].head(num_recommendations)
    
    return hidden_gems

# Contoh: Rekomendasi untuk User ID nomor 1
print("Rekomendasi Hidden Gems untuk User 1:")
display(rekomendasi_hidden_gems(user_id=1))

Rekomendasi Hidden Gems untuk User 1:


,movieId,prediksi_rating,title,genres,avg_rating,vote_count
28,1375,1.937626,Star Trek III: The Search for Spock (1984),Action|Adventure|Sci-Fi,3.261905,42
30,1376,1.902531,Star Trek IV: The Voyage Home (1986),Adventure|Comedy|Sci-Fi,3.476744,43
38,2003,1.752438,Gremlins (1984),Comedy|Horror,3.378049,41
40,1129,1.749285,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,3.448718,39
44,1909,1.710080,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,3.440476,42


In [13]:
# Mengambil 5 ID user pertama sebagai contoh
sample_users = [1, 2, 3, 4, 5]

for uid in sample_users:
    print(f"\n" + "="*50)
    print(f"REKOMENDASI HIDDEN GEMS UNTUK USER ID: {uid}")
    print("="*50)
    
    # Memanggil fungsi rekomendasi yang sudah kita buat di Cell 12
    hasil_rekom = rekomendasi_hidden_gems(user_id=uid, num_recommendations=5)
    
    if not hasil_rekom.empty:
        display(hasil_rekom[['title', 'genres', 'prediksi_rating', 'vote_count']])
    else:
        print("User ini sudah menonton terlalu banyak film atau data tidak cukup.")


REKOMENDASI HIDDEN GEMS UNTUK USER ID: 1


,title,genres,prediksi_rating,vote_count
28,Star Trek III: The Search for Spock (1984),Action|Adventure|Sci-Fi,1.937626,42
30,Star Trek IV: The Voyage Home (1986),Adventure|Comedy|Sci-Fi,1.902531,43
38,Gremlins (1984),Comedy|Horror,1.752438,41
40,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,1.749285,39
44,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,1.710080,42



REKOMENDASI HIDDEN GEMS UNTUK USER ID: 2


,title,genres,prediksi_rating,vote_count
15,The Martian (2015),Adventure|Drama|Sci-Fi,0.631728,48
36,Gone Girl (2014),Drama|Thriller,0.494268,37
37,Gran Torino (2008),Crime|Drama,0.493716,46
44,Edge of Tomorrow (2014),Action|Sci-Fi|IMAX,0.453419,44
46,The Revenant (2015),Adventure|Drama,0.442816,31



REKOMENDASI HIDDEN GEMS UNTUK USER ID: 3


,title,genres,prediksi_rating,vote_count
16,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,0.152513,39
23,Superman II (1980),Action|Sci-Fi,0.137285,49
25,"NeverEnding Story, The (1984)",Adventure|Children|Fantasy,0.135504,43
26,"Fly, The (1986)",Drama|Horror|Sci-Fi|Thriller,0.134980,43
27,Star Trek IV: The Voyage Home (1986),Adventure|Comedy|Sci-Fi,0.129725,43



REKOMENDASI HIDDEN GEMS UNTUK USER ID: 4


,title,genres,prediksi_rating,vote_count
20,Raging Bull (1980),Drama,1.744021,40
22,"Player, The (1992)",Comedy|Crime|Drama,1.737876,38
27,"Manchurian Candidate, The (1962)",Crime|Thriller|War,1.666866,30
32,"Maltese Falcon, The (1941)",Film-Noir|Mystery,1.556413,44
36,Sling Blade (1996),Drama,1.507751,47



REKOMENDASI HIDDEN GEMS UNTUK USER ID: 5


,title,genres,prediksi_rating,vote_count
36,French Kiss (1995),Action|Comedy|Romance,0.811925,45
40,Nell (1994),Drama,0.739509,40
43,Don Juan DeMarco (1995),Comedy|Drama|Romance,0.707794,40
45,Rob Roy (1995),Action|Drama|Romance|War,0.698653,44
46,"Specialist, The (1994)",Action|Drama|Thriller,0.686091,38


In [14]:
# DI MAIN_SKRIPSI.IPYNB (CELL PALING AKHIR)
import pandas as pd

try:
    # Kita buat dataframe baru yang mengaitkan movieId, title, dan prediksi_rating secara kokoh
    df_pred_user = df_prediksi.loc[1].reset_index()
    df_pred_user.columns = ['movieId', 'prediksi_rating']
    
    # Gabungkan dengan df_movies bawaan MovieLens untuk mengambil judul aslinya
    df_export_svd = pd.merge(df_pred_user, df_movies[['movieId', 'title']], on='movieId', how='inner')
    
    # Ekspor kolom title dan prediksi_rating secara bersih tanpa spasi hantu
    df_export_svd['title'] = df_export_svd['title'].str.replace(r'\s\(\d{4}\)', '', regex=True).str.strip()
    df_export_svd = df_export_svd[['title', 'prediksi_rating']]
    
    # Simpan ke CSV
    df_export_svd.to_csv('pengetahuan_svd.csv', index=False)
    print("=== [BERHASIL 100%] File 'pengetahuan_svd.csv' telah diekspor dengan kolom TITLE yang valid! ===")
    print("Silakan jalankan ulang Cell ini, lalu kembali ke second_skripsi.ipynb.")
except Exception as e:
    print(f"❌ Gagal melakukan ekspor: {e}")

=== [BERHASIL 100%] File 'pengetahuan_svd.csv' telah diekspor dengan kolom TITLE yang valid! ===
Silakan jalankan ulang Cell ini, lalu kembali ke second_skripsi.ipynb.


In [15]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Load Data
df = pd.read_csv("dataset/ml-latest-small/ratings.csv")
user_item_matrix = df.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# 2. Train-Test Split (Sesuai poin 3.4.7 lu)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Implementasi SVD dengan Regularisasi (TruncatedSVD)
# Di sklearn, regularisasi sudah otomatis ditangani oleh solver 'arpack' atau 'randomized'
svd_model = TruncatedSVD(n_components=20, n_iter=7, random_state=42)
matrix_factors = svd_model.fit_transform(user_item_matrix)

# 4. Bukti Pencegahan Overfitting (Cek Explained Variance)
print(f"Total Variance yang dijelaskan: {svd_model.explained_variance_ratio_.sum():.2f}")
print("Interpretasi: Model tidak overfitting jika variance yang ditangkap tidak 100% (noise tidak ikut dipelajari).")

Total Variance yang dijelaskan: 0.39
Interpretasi: Model tidak overfitting jika variance yang ditangkap tidak 100% (noise tidak ikut dipelajari).
